In [25]:
using LowLevelFEM

# Axisymmetric Linear Elasticity

This example demonstrates the solution of a two-dimensional axisymmetric
linear elasticity problem using LowLevelFEM.

The problem is solved in two different ways:

1. using the built-in axisymmetric elasticity solver,
2. using the weak-form DSL.

The two solutions are then compared to verify the correctness of the
DSL implementation.

In [26]:
structured_rect_mesh(x0=20, lx=80, ly=20, n=10, order=2)
mat = Material("body");

## Geometry and material

A quadratic structured mesh is generated for a hollow cylinder.
The inner radius avoids the singularity associated with the axis of
revolution.

Linear isotropic elastic material properties are assigned to the body.

## Built-in axisymmetric formulation

First, the problem is solved using the dedicated axisymmetric elasticity
implementation available in LowLevelFEM.

The right boundary is subjected to a radial traction, while the lower-left
corner is constrained to remove rigid body motion.

In [27]:
prob = Problem([mat], type=:AxiSymmetric)

ld = load("left", fx=1);

In [28]:
bc = BoundaryCondition("leftbottom", uy=0);

In [29]:
u1 = solveDisplacement(prob, load=[ld], support=[bc])

nodal VectorField
[0.00013832785258954998; 0.0; … ; 4.179436241038414e-5; -2.383797343002192e-6;;]

In [30]:
S1 = solveStress(u1)

elementwise TensorField
[[-0.9919574037127676; -0.0008989088111019394; … ; 0.0; 0.9849927923882467;;], [-0.9919169282987464; -2.1214958242333942e-5; … ; 0.0; 0.984874902779041;;], [-0.9920190304455192; 2.523447196393166e-7; … ; 0.0; 0.9848770822377156;;], [-0.9920315525528828; -3.6425744398602775e-8; … ; 0.0; 0.9848787917587428;;], [-0.9920329505298087; -5.052705716430854e-8; … ; 0.0; 0.9848793201249366;;], [-0.9920330982980252; -2.2870973778038595e-8; … ; 0.0; 0.9848793201249537;;], [-0.9920328067102262; 7.865812118232767e-8; … ; 0.0; 0.9848787917587805;;], [-0.9920303332614276; 1.4889026927099206e-6; … ; 0.0; 0.9848770822377876;;], [-0.9920039897761425; 2.3956517709092666e-5; … ; 0.0; 0.9848749027791375;;], [-0.9917117855027554; 0.000370368100729923; … ; 0.0; 0.9849927923883657;;]  …  [-0.001704365321434198; -2.8734470450828345e-6; … ; 0.0; 0.0841762424185871;;], [-0.0017030046919840822; -4.881511861398735e-8; … ; 0.0; 0.08417598509437091;;], [-0.0017027068421273522; 2.01057046695913

In [31]:
showDoFResults(u1, name="u1", visible=true, factor=1e5)
showStressResults(S1, name="σ1 - von Mises");

## Reference solution

The displacement and von Mises stress obtained from the built-in
axisymmetric formulation will be used as the reference solution.

## Weak-form DSL formulation

The same problem is assembled using the LowLevelFEM weak-form DSL.

The axisymmetric strain-displacement operator is written directly from
its mathematical definition.

In [32]:
Pu = Problem([mat], type=:VectorField, dim=2, field=:u);

In [33]:
r = ScalarField(Pu, "body", (x, y, z)->x)

elementwise ScalarField
[[20.0; 22.0; … ; 20.0; 21.0;;], [20.0; 21.999999999999993; … ; 20.0; 21.0;;], [20.0; 22.0; … ; 20.0; 21.0;;], [20.0; 22.0; … ; 20.0; 21.0;;], [20.0; 22.0; … ; 20.0; 21.0;;], [20.0; 22.0; … ; 20.0; 21.0;;], [20.0; 22.0; … ; 20.0; 21.0;;], [20.0; 22.0; … ; 20.0; 21.0;;], [20.0; 22.0; … ; 20.0; 21.0;;], [20.0; 21.999999999999993; … ; 20.0; 21.0;;]  …  [98.0; 100.0; … ; 98.0; 99.0;;], [98.0; 100.0; … ; 98.0; 99.0;;], [98.0; 100.0; … ; 98.0; 99.0;;], [98.0; 100.0; … ; 98.0; 99.0;;], [98.0; 100.0; … ; 98.0; 99.0;;], [98.0; 100.0; … ; 98.0; 99.0;;], [98.0; 100.0; … ; 98.0; 99.0;;], [98.0; 100.0; … ; 98.0; 99.0;;], [98.0; 100.0; … ; 98.0; 99.0;;], [98.0; 100.0; … ; 98.0; 99.0;;]]

### Axisymmetric strain operator

The strain vector is written as

$$
\varepsilon =
\begin{bmatrix}
\partial u_r/\partial r \\
u_r/r \\
\partial u_z/\partial z \\
\partial u_r/\partial z + \partial u_z/\partial r
\end{bmatrix},
$$

which is represented as

$$
\varepsilon = A_1 \nabla^s u + A_2 u.
$$

This decomposition allows the operator to be expressed naturally in the DSL.

In [34]:
A1 = [1 0 0; 0 0 0; 0 1 0; 0 0 1]

4×3 Matrix{Int64}:
 1  0  0
 0  0  0
 0  1  0
 0  0  1

In [35]:
A2 = [0 0; 1/r 0; 0 0; 0 0]

4×2 Matrix{Any}:
 0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [36]:
B = A1 ⋅ SymGrad(Pu) + A2 ⋅ Pu;

In [37]:
E = mat.E
ν = mat.ν

D = E / (1+ν) / (1-2ν) * [1-ν ν ν 0; ν 1-ν ν 0; ν ν 1-ν 0; 0 0 0 (1-2ν)/2]

4×4 Matrix{Float64}:
 2.69231e5  1.15385e5  1.15385e5      0.0
 1.15385e5  2.69231e5  1.15385e5      0.0
 1.15385e5  1.15385e5  2.69231e5      0.0
 0.0        0.0        0.0        76923.1

### System assembly

The global stiffness matrix is assembled from the weak form

$$
K = \int_\Omega B^\mathrm{T} D B \; 2\pi r \; d\Omega.
$$

The factor $2\pi r$ accounts for the axisymmetric volume measure.

In [38]:
K2 = ∫(B' ⋅ D ⋅ B * (2π*r))
K2[:, :]

3402×3402 SparseArrays.SparseMatrixCSC{Float64, Int64} with 104003 stored entries:
⎡⢿⣷⡍⠈⠩⠭⢷⣒⣒⣒⣲⣤⠬⠭⠅⠭⠅⠿⠆⢶⣒⣒⣒⣒⢐⣒⢐⣒⢐⣒⢐⣶⢰⣤⢠⠤⠤⠤⠬⠍⎤
⎢⡃⠉⣻⣾⣄⡤⠤⠤⢤⣶⣒⣛⣛⣣⡤⠤⠤⠤⠄⠤⠄⠤⠄⠤⠄⢤⣤⣴⣶⣖⣒⣒⣒⣚⢘⣛⢙⣛⢙⣛⎥
⎢⡇⡆⠀⡽⡿⣯⡉⠉⠉⠁⠀⠀⠀⠿⣯⣭⣉⡉⠁⠉⠁⠉⠁⠉⠁⠉⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢹⢳⠀⡇⡇⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠈⠉⠙⠛⠶⢶⣤⣄⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢸⢸⢠⣷⠇⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠛⠳⠶⣤⣤⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠘⣾⣼⢸⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠛⠛⠶⢦⣤⣄⣀⠀⠀⠀⎥
⎢⡆⡇⠿⣸⣤⡄⠀⠀⠀⠀⠀⠈⠻⣦⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠛⠷⠶⎥
⎢⡅⡅⠀⡏⡏⣿⡀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣥⡅⠀⡇⡇⠸⣇⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢨⣅⠀⡅⡅⠀⢻⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢸⢸⠀⡅⡅⠀⠘⣷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢸⢸⠀⡅⡅⠀⠀⢹⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢰⢰⠀⣅⡅⠀⠀⠀⢿⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢰⢰⢀⣿⠃⠀⠀⠀⠘⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢰⢰⢸⢿⠀⠀⠀⠀⠀⢻⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢰⣴⢸⢸⠀⠀⠀⠀⠀⠈⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠐⣶⣸⢸⠀⠀⠀⠀⠀⠀⠸⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⎥
⎢⠀⡖⣶⢰⠀⠀⠀⠀⠀⠀⠀⢿⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⎥
⎢⠀⡇⣷⢰⠀⠀⠀⠀⠀⠀⠀⠘⣷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⎥
⎣⡆⠇⣷⢰⠀⠀⠀⠀⠀⠀⠀⠀⢹⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⎦

In [39]:
f2 = ∫(Pu ⋅ [1, 0] * (2π*r), Γ="left")

nodal VectorField
[41.88790204786388; 0.0; … ; 0.0; 0.0;;]

## Solution

The linear system assembled from the DSL is solved using the same boundary
conditions as the reference formulation.

In [40]:
u2 = solveField(K2, f2, support=[bc])

nodal VectorField
[0.00013832760921313298; 0.0; … ; 4.17943301292467e-5; -2.3840243020994944e-6;;]

In [41]:
showDoFResults(u2, name="u2", visible=true, factor=1e5);

## Stress recovery

The strain components are computed from the displacement field,
followed by Hooke's law to obtain the stress tensor.

Finally, the von Mises equivalent stress is evaluated.

In [42]:
εr = ∂x(u2[1])
εφ = u2[1] / r
εz = ∂y(u2[2])
γrz = ∂y(u2[1]) + ∂x(u2[2]);

In [43]:
ε = [εr, εφ, εz, γrz]

4-element Vector{ScalarField}:
 ScalarField([[-6.591217447531029e-6; -5.423075942294559e-6; … ; -6.5963085033882976e-6; -6.009968650353995e-6;;], [-6.5947767333009535e-6; -5.4239103257396045e-6; … ; -6.59547942937432e-6; -6.009468470094439e-6;;], [-6.595414138549165e-6; -5.423441805599977e-6; … ; -6.595485902122764e-6; -6.009434855346532e-6;;], [-6.595490422610373e-6; -5.42338022132638e-6; … ; -6.5954998180455405e-6; -6.009436847405746e-6;;], [-6.5955031153106165e-6; -5.423373178424257e-6; … ; -6.595504771000958e-6; -6.00943865280191e-6;;], [-6.595505553326009e-6; -5.423372470586118e-6; … ; -6.595504771000796e-6; -6.0094386528018425e-6;;], [-6.595503115310454e-6; -5.423373178424203e-6; … ; -6.595499818045324e-6; -6.009436847405515e-6;;], [-6.595490422609993e-6; -5.423380221326055e-6; … ; -6.595485902122331e-6; -6.009434855346125e-6;;], [-6.595414138548569e-6; -5.423441805599651e-6; … ; -6.595479429373507e-6; -6.009468470093938e-6;;], [-6.594776733299761e-6; -5.423910325739437e-6; … ; -

In [44]:
σ = D * ε

4-element Vector{ScalarField}:
 ScalarField([[-0.9919879673124865; -0.8127054007870965; … ; -0.9927372872784843; -0.906875921718949;;], [-0.9919472277852721; -0.812214703217997; … ; -0.9920929070360834; -0.9069194118632221;;], [-0.9920493988140725; -0.8120936390111283; … ; -0.9920640499609634; -0.9069112355331127;;], [-0.9920619549874159; -0.8120793599259084; … ; -0.9920634141071198; -0.9069094309774401;;], [-0.99206336981251; -0.8120773899979714; … ; -0.9920635700011373; -0.9069090210020103;;], [-0.9920635218731786; -0.8120770942100198; … ; -0.9920635700011189; -0.9069090210020145;;], [-0.9920632227072061; -0.8120774834085673; … ; -0.9920634141071367; -0.906909430977444;;], [-0.9920607215831141; -0.8120802600201961; … ; -0.9920640499609753; -0.9069112355331153;;], [-0.9920342757073091; -0.8120967564936936; … ; -0.9920929070360841; -0.9069194118632415;;], [-0.9917419680839652; -0.8120118059312498; … ; -0.9927372872786391; -0.9068759217189349;;]  …  [-0.001704386147979825; 1.51831185729

In [45]:
σr = σ[1]
σφ = σ[2]
σz = σ[3]
τrz = σ[4];

In [46]:
σeqv2 = √(((σr - σφ) * (σr - σφ) + (σφ - σz) * (σφ - σz) + (σz - σr) * (σz - σr)) / 2 + 3τrz * τrz)

elementwise ScalarField
[[1.8002598484689716; 1.488182735023767; … ; 1.8009482605848643; 1.6389006183392978;;], [1.8007299799696346; 1.4882676963504666; … ; 1.8008231852831258; 1.6388429175305599;;], [1.80081316430263; 1.4882070842749304; … ; 1.8008224772383892; 1.6388380873291257;;], [1.8008228545405565; 1.4881989564467626; … ; 1.8008240063145131; 1.6388380704844738;;], [1.800824372266652; 1.4881979186568342; … ; 1.8008245602002635; 1.6388382000813444;;], [1.8008246488738566; 1.4881977859836701; … ; 1.800824560200218; 1.6388382000813129;;], [1.8008243674808075; 1.4881979223152948; … ; 1.800824006314411; 1.6388380704843748;;], [1.8008228144131884; 1.4881989917039113; … ; 1.8008224772382082; 1.638838087328957;;], [1.800812672672339; 1.4882072064476008; … ; 1.8008231852828445; 1.63884291753033;;], [1.8007234142512447; 1.4882597873525358; … ; 1.8009482605844869; 1.6389006183389903;;]  …  [0.08591973450016341; 0.08332965539535916; … ; 0.08591900476137654; 0.08460740790782866;;], [0.0859187

In [47]:
showElementResults(σeqv2, name="σ2 - von Mises");

## Verification

The displacement and stress fields obtained from the weak-form DSL agree
with those produced by the built-in axisymmetric formulation, confirming
the correctness of the DSL implementation.

In [48]:
openPostProcessor()